# Customer Segmentation Visualization

### Importing Data

In [5]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns

customer_segments = pd.read_csv('../data/processed/customer_segments.csv')
customer_segments.head()

,Segment,AvgTotalSpent,TotalSpentCount,AvgPurchaseFrequency,AvgCustomerLifespan,SegmentLabel
0,0,360.38,872,31.11,1.02,BudgetRegular
1,1,332.74,752,28.40,5.26,BudgetIrregular
2,2,2377.40,673,1.26,241.04,HighValueIrregular
3,3,779.54,1607,1.02,169.69,HighSpendingIrregular


### 1.Segment Size Distribution

In [8]:
segment_sizes = customer_segments['SegmentLabel'].value_counts()

figure = go.Figure(
  go.Pie(
    labels=segment_sizes.index,
    values=segment_sizes.values,
    textinfo='percent',
    hole=0.3
  )
)

figure.show()

In [10]:
customer_segments.head()

,Segment,AvgTotalSpent,TotalSpentCount,AvgPurchaseFrequency,AvgCustomerLifespan,SegmentLabel
0,0,360.38,872,31.11,1.02,BudgetRegular
1,1,332.74,752,28.40,5.26,BudgetIrregular
2,2,2377.40,673,1.26,241.04,HighValueIrregular
3,3,779.54,1607,1.02,169.69,HighSpendingIrregular


### 2.Segment Value Distribution

In [11]:
segment_value = customer_segments.groupby('SegmentLabel').agg({
  'AvgTotalSpent': 'sum',
  'SegmentLabel': 'count'
})

figure = go.Figure(
  go.Bar(
    x=segment_value['SegmentLabel'],
    y=segment_value['AvgTotalSpent'],
    text=(segment_value['AvgTotalSpent']/1000).round(1),
    textposition='auto',
    name='Segment Revenue (K)'
  )
)

figure.show()

### 3.Segment Characteristics

In [12]:
segment_chars = customer_segments.groupby('Segment').agg({
  'avg_order_value': 'mean',
  'purchase_frequency_mean': 'mean',
  'customer_lifespan_mean': 'mean'
}).round(2)

segment_chars_norm = (segment_chars - segment_chars.min()) / (segment_chars.max() - segment_chars.min())

figure = make_subplots(rows=2, cols=2, specs=[[{'type': 'polar'}, {'type': 'polar'}], [{'type': 'polar'}, {'type': 'polar'}]])


figure.add_trace(
  go.Scatterpolar(
    r=segment_chars_norm.iloc[0].values,
    theta=segment_chars_norm.columns.to_list(),
    name="High Value Irregular",
    fill='toself'
  ),
  row=1, col=1
)

figure.add_trace(
  go.Scatterpolar(
    r=segment_chars_norm.iloc[1].values,
    theta=segment_chars_norm.columns.to_list(),
    name='Low Value Irregular',
    fill='toself'
  ),
  row=1, col=2
)

figure.add_trace(
  go.Scatterpolar(
    r=segment_chars_norm.iloc[2].values,
    theta=segment_chars_norm.columns.to_list(),
    name="Low Value Regular",
    fill='toself'
  ),
  row=2, col=1
)

figure.add_trace(
  go.Scatterpolar(
    r=segment_chars_norm.iloc[3].values,
    theta=segment_chars_norm.columns.to_list(),
    name="High Value Regular",
    fill='toself'
  ),
  row=2, col=2
)

figure.update_layout(
  polar = dict(
      radialaxis_tickfont_size = 8,
      angularaxis = dict(
        tickfont_size = 8,
        rotation = 90,
        direction = "clockwise"
      )
    ),
    polar2 = dict(
      radialaxis_tickfont_size = 8,
      angularaxis = dict(
        tickfont_size = 8,
        rotation = 90,
        direction = "clockwise"
      )
    ),
    polar3 = dict(
      radialaxis_tickfont_size = 8,
      angularaxis = dict(
        tickfont_size = 8,
        rotation = 90,
        direction = "clockwise"
      )
    ),
    polar4 = dict(
      radialaxis_tickfont_size = 8,
      angularaxis = dict(
        tickfont_size = 8,
        rotation = 90,
        direction = "clockwise"
      )
    )
)

figure.show()

KeyError: "Column(s) ['avg_order_value', 'customer_lifespan_mean', 'purchase_frequency_mean'] do not exist"

### 4.Segment Purchase Patterns

In [6]:
customer_segments['first_purchase'] = pd.to_datetime(customer_segments['first_purchase'])
customer_segments['last_purchase'] = pd.to_datetime(customer_segments['last_purchase'])
customer_segments['days_between_purchases'] = (customer_segments['last_purchase'] - customer_segments['first_purchase'] ).dt.days

purchase_patterns = customer_segments.groupby('Segment').agg({
  'days_between_purchases': 'mean',
  'purchase_frequency_mean': 'mean'
}).round(2)

figure = go.Figure(
  go.Scatter(
    x=purchase_patterns['days_between_purchases'],
    y=purchase_patterns['purchase_frequency_mean'],
    mode='markers+text',
    text=purchase_patterns.index,
    textposition='top center',
    marker=dict(size=15),
    name='Purchase Patterns'
  )
)

figure.show()

### 5.Segment Lifespan Analysis

In [14]:
lifespan_data = customer_segments.groupby('Segment_Label')['customer_lifespan'].agg(['mean', 'std']).reset_index()

figure = go.Figure(
  go.Bar(
    x=lifespan_data['Segment_Label'],
    y=lifespan_data['mean'],
    error_y=dict(
      type='data',
      array=lifespan_data['std'],
      color='red'
    ),
    name='Customer Avg Lifespan'
  )
)

figure.show()